# AgriVoice: N-ATLaS on Kaggle

Runs the official [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) on Kaggle's free GPUs and connects it to the AgriVoice backend.

## One-time setup
1. **Settings → Accelerator → GPU T4 x2** (N-ATLaS needs both GPUs).
2. **Settings → Internet → On** (requires a phone-verified Kaggle account).
3. **Add-ons → Secrets**, add and attach to this notebook:
   - `HF_TOKEN`: Hugging Face token of an account that accepted the N-ATLaS licence
   - `AGRIVOICE_URL`: your backend, e.g. `https://your-username-agrivoice.hf.space` *(optional)*
   - `AGRIVOICE_ADMIN_TOKEN`: the backend's `ADMIN_TOKEN` *(optional)*

   Secrets only work for runs started from the Kaggle website. For runs started
   through the Kaggle API, put the same values in a **private** dataset named
   `agrivoice-secrets`, one file per secret (`hf_token`, `agrivoice_url`,
   `agrivoice_admin_token`), and attach it to this notebook.

## Each session
- **Easiest:** press **Start** in the AgriVoice admin dashboard (N-ATLaS tab). It runs this notebook for you and **Stop** shuts it down. Needs the private `agrivoice-secrets` dataset.
- **By hand:** *Save Version → Save & Run All (Commit)*. Set `RUN_HOURS` in the last cell first (default 2; ~9 on review days).

The last cell prints the public N-ATLaS link and registers it with the backend. Without backend settings, paste the link into the backend's `/admin` page.

In [ ]:
# Filled in by the AgriVoice backend when it starts this notebook from /admin.
# Leave it empty for runs started by hand (Kaggle Secrets are used instead).
RUN_CONFIG = {}

In [ ]:
!pip install -q "gradio==6.29.0" spaces "transformers==5.17.0" accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

import glob

_secrets = UserSecretsClient()

def _secret(name):
    # 1) Kaggle Secrets: only reachable when the run is started from the website
    try:
        value = _secrets.get_secret(name)
        if value:
            return value
    except Exception:
        pass
    # 2) Private dataset "agrivoice-secrets" holding one file per secret,
    #    used for runs started through the Kaggle API
    for path in glob.glob(f"/kaggle/input/**/agrivoice-secrets/{name.lower()}", recursive=True):
        value = open(path).read().strip()
        if value:
            return value
    return None

os.environ["HF_TOKEN"] = _secret("HF_TOKEN") or ""
os.environ["DEVICE"] = "auto"  # spread N-ATLaS across both T4 GPUs
AGRIVOICE_URL = (RUN_CONFIG.get("backend_url") or _secret("AGRIVOICE_URL") or "").rstrip("/")
AGRIVOICE_ADMIN_TOKEN = None if RUN_CONFIG else _secret("AGRIVOICE_ADMIN_TOKEN")
assert os.environ["HF_TOKEN"], "Add the HF_TOKEN secret, or the private agrivoice-secrets dataset."
print("Started from:", "AgriVoice dashboard" if RUN_CONFIG else "Kaggle website",
      "| Backend:", AGRIVOICE_URL or "none")

In [ ]:
import glob, os, subprocess

def _find(name):
    hits = glob.glob(f"/kaggle/input/**/agrivoice-tts-assets/**/{name}", recursive=True) or \
           glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

ckpt = _find("wavtokenizer_large_speech_320_24k.ckpt")
config = _find("wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml")
if ckpt and config:
    # outetts without its llama-cpp dependency (only its audio decoder is used)
    subprocess.run("pip install -q --no-deps outetts==0.3.3 && pip install -q uroman inflect einops soundfile",
                   shell=True, check=True)
    if not os.path.isdir("/kaggle/working/yarngpt"):
        subprocess.run("git clone -q --depth 1 https://github.com/saheedniyi02/yarngpt.git /kaggle/working/yarngpt",
                       shell=True, check=True)
    os.environ.update(TTS_ENABLED="1", YARNGPT_DIR="/kaggle/working",
                      WAVTOKENIZER_CKPT=ckpt, WAVTOKENIZER_CONFIG=config)
    print("Text-to-speech: on")
else:
    print("Text-to-speech: off (attach the agrivoice-tts-assets dataset to enable it)")

In [ ]:
%%writefile tts.py
"""
Text-to-speech with YarnGPT2 (Nigerian voices for Yoruba, Hausa, Igbo, English).

Runs next to N-ATLaS on the GPU. An answer is split into sentences, which are
generated together in one batch (one sequence each) and joined with short
pauses, so a whole answer takes about as long as its longest sentence.

Needs the YarnGPT repository (for `yarngpt.audiotokenizer` and its voice
files) and the WavTokenizer decoder checkpoint + config that YarnGPT uses.
https://github.com/saheedniyi02/yarngpt · https://huggingface.co/saheedniyi/YarnGPT2
"""
from __future__ import annotations

import importlib.util
import re
import sys
import types
from typing import Any

import numpy as np
import torch

MODEL_ID = "saheedniyi/YarnGPT2"
SAMPLE_RATE = 24000

# AgriVoice language code → YarnGPT language and default voice
LANGUAGES = {"en-ng": "english", "yo": "yoruba", "ha": "hausa", "ig": "igbo"}
DEFAULT_VOICES = {
    "en-ng": "idera",
    "yo": "yoruba_female2",
    "ha": "hausa_female1",
    "ig": "igbo_female2",
}

_MAX_WORDS = 22      # longer sentences are split; very long ones degrade
_BATCH_SIZE = 8
_PAUSE_SECONDS = 0.3


def split_sentences(text: str) -> list[str]:
    """Plain-text sentences of at most _MAX_WORDS words, without Markdown."""
    text = re.sub(r"\*\*|__|`|#+\s*", "", text)               # bold, code, headings
    text = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s+", "", text, flags=re.M)  # list markers
    parts = re.split(r"(?<=[.!?;:])\s+|\n+", text)
    sentences: list[str] = []
    for part in (p.strip() for p in parts):
        words = part.split()
        for i in range(0, len(words), _MAX_WORDS):
            chunk = " ".join(words[i : i + _MAX_WORDS])
            if re.search(r"[A-Za-zÀ-ɏḀ-ỿ]", chunk):
                sentences.append(chunk)
    return sentences


def _load_outetts_decoder_only() -> None:
    """
    YarnGPT imports outetts.wav_tokenizer (the audio decoder). Importing it
    normally runs outetts/__init__.py, which pulls in its whole TTS stack
    (llama-cpp, MeCab, Whisper, …) that YarnGPT never uses. Registering the
    package without running __init__ makes only the decoder load.
    """
    if "outetts" in sys.modules:
        return
    spec = importlib.util.find_spec("outetts")
    if spec is None or not spec.submodule_search_locations:
        raise ImportError("outetts is not installed (pip install --no-deps outetts==0.3.3)")
    package = types.ModuleType("outetts")
    package.__path__ = list(spec.submodule_search_locations)
    sys.modules["outetts"] = package


class YarnTTS:
    def __init__(self, yarngpt_parent_dir: str, wavtokenizer_ckpt: str, wavtokenizer_config: str) -> None:
        if yarngpt_parent_dir not in sys.path:
            sys.path.insert(0, yarngpt_parent_dir)
        _load_outetts_decoder_only()
        from transformers import AutoModelForCausalLM  # noqa: PLC0415
        from yarngpt.audiotokenizer import AudioTokenizerV2  # type: ignore[import-not-found]  # noqa: PLC0415

        self.audio_tokenizer: Any = AudioTokenizerV2(MODEL_ID, wavtokenizer_ckpt, wavtokenizer_config)
        self.device = self.audio_tokenizer.device
        self.model: Any = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto").to(self.device).eval()
        tokenizer = self.audio_tokenizer.tokenizer
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

    def speak(self, text: str, language: str, voice: str | None = None) -> np.ndarray:
        """Float32 mono audio at SAMPLE_RATE for *text* in *language* (yo/ha/ig/en-ng)."""
        if language not in LANGUAGES:
            raise ValueError(f"Unsupported language: {language}")
        sentences = split_sentences(text)
        if not sentences:
            raise ValueError("Nothing to say.")
        voice = voice or DEFAULT_VOICES[language]
        pause = np.zeros(int(SAMPLE_RATE * _PAUSE_SECONDS), dtype=np.float32)
        pieces: list[np.ndarray] = []
        for start in range(0, len(sentences), _BATCH_SIZE):
            for audio in self._batch(sentences[start : start + _BATCH_SIZE], LANGUAGES[language], voice):
                pieces += [audio, pause]
        return np.concatenate(pieces[:-1])

    def _batch(self, sentences: list[str], yarn_language: str, voice: str) -> list[np.ndarray]:
        tok = self.audio_tokenizer
        prompts = [tok.create_prompt(s, lang=yarn_language, speaker_name=voice) for s in sentences]
        enc = tok.tokenizer(
            prompts, add_special_tokens=False, padding=True, padding_side="left", return_tensors="pt"
        ).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **enc,
                do_sample=True,
                temperature=0.1,           # YarnGPT's recommended settings
                repetition_penalty=1.1,
                max_new_tokens=2500,       # ~30 s of audio per sentence
                pad_token_id=tok.tokenizer.pad_token_id,
            )
        prompt_length = enc["input_ids"].shape[1]
        audios: list[np.ndarray] = []
        for row in out:
            codes = tok.extract_integers(tok.tokenizer.decode(row[prompt_length:]))
            if not codes:
                continue
            audio = tok.get_audio(codes)
            audios.append(audio.squeeze().float().cpu().numpy().astype(np.float32))
        return audios

In [ ]:
%%writefile app.py
"""
N-ATLaS on Hugging Face ZeroGPU.

Follows the official usage on https://huggingface.co/NCAIR1/N-ATLaS:
  - AutoTokenizer + AutoModelForCausalLM, float16
  - apply_chat_template(..., tokenize=False, date_string=<today>)
  - tokenizer(text, add_special_tokens=False)
  - generate(max_new_tokens, use_cache=True, repetition_penalty=1.12, greedy)

Two differences from the model card's snippet, both needed to run it:
  - `from datetime import datetime` (missing in the card)
  - decode only the newly generated tokens, so the prompt isn't echoed back

The AgriVoice backend calls the `/generate` endpoint with a list of chat
messages. The "Chat" tab is for trying the model by hand.

With TTS_ENABLED=1, `/speak` also reads text aloud with YarnGPT2 (tts.py),
on the same GPU. It needs YARNGPT_DIR, WAVTOKENIZER_CKPT, WAVTOKENIZER_CONFIG.
"""
from __future__ import annotations

import os
import tempfile
from datetime import datetime

import gradio as gr
import spaces
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "NCAIR1/N-ATLaS")
# "cuda": ZeroGPU (model placed on CUDA at import time, as ZeroGPU requires)
# "auto": spread across several GPUs, e.g. Kaggle's 2 × T4 (16 GB doesn't fit one)
# "cpu":  local testing with a small stand-in model only
DEVICE = os.getenv("DEVICE", "cuda")
HF_TOKEN = os.getenv("HF_TOKEN")  # Space secret; N-ATLaS is a gated model

DEFAULT_MAX_NEW_TOKENS = 512
MAX_NEW_TOKENS_LIMIT = 1000  # The model card's value
REPETITION_PENALTY = 1.12

DEFAULT_SYSTEM_PROMPT = (
    "you are a large language model trained by Awarri AI technologies. "
    "You are a friendly assistant and you are here to help."
)

_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if _tokenizer is None:
    raise RuntimeError(f"Could not load the tokenizer for {MODEL_ID}")
tokenizer = _tokenizer

if DEVICE == "auto":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, token=HF_TOKEN, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32 if DEVICE == "cpu" else torch.float16,
        token=HF_TOKEN,
    ).to(DEVICE)
model.eval()

# Optional text-to-speech, loaded once like the model (off unless TTS_ENABLED=1)
tts_engine = None
if os.getenv("TTS_ENABLED") == "1":
    from tts import YarnTTS  # noqa: PLC0415

    tts_engine = YarnTTS(
        os.environ["YARNGPT_DIR"], os.environ["WAVTOKENIZER_CKPT"], os.environ["WAVTOKENIZER_CONFIG"]
    )


def format_text_for_inference(messages: list[dict[str, str]]) -> str:
    current_date = datetime.now().strftime("%d %b %Y")
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        date_string=current_date,
    )
    if not isinstance(text, str):  # tokenize=False always returns a string
        raise TypeError(f"Unexpected chat template output: {type(text).__name__}")
    return text


@spaces.GPU(duration=90)
def generate(messages: list[dict[str, str]], max_new_tokens: float = DEFAULT_MAX_NEW_TOKENS) -> str:
    """
    Generate the assistant's reply.

    Args:
        messages: [{"role": "system" | "user" | "assistant", "content": "..."}, ...]
        max_new_tokens: Upper bound on reply length (capped at 1000). A float because
            Gradio sliders and API clients may send e.g. 512.0.
    """
    if not messages or not all(
        isinstance(m, dict) and m.get("role") and isinstance(m.get("content"), str)
        for m in messages
    ):
        raise gr.Error("`messages` must be a list of {role, content} objects.")

    max_new_tokens = max(1, min(int(max_new_tokens), MAX_NEW_TOKENS_LIMIT))
    text = format_text_for_inference(messages)
    input_tokens = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **input_tokens,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            repetition_penalty=REPETITION_PENALTY,
            # The card passes temperature=0.1 without do_sample, which decodes
            # greedily; this is the same behaviour stated explicitly.
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_tokens["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    if not isinstance(reply, str):  # one sequence decodes to one string
        raise TypeError(f"Unexpected decode output: {type(reply).__name__}")
    return reply.strip()


def _text(content: object) -> str:
    """Gradio 6 chat history may hold content as a list of parts."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            part.get("text", "") if isinstance(part, dict) else str(part) for part in content
        )
    return str(content)


@spaces.GPU(duration=120)
def speak(text: str, language: str, voice: str = "") -> str:
    """
    Read *text* aloud. Returns the path of a 24 kHz WAV file.

    Args:
        text: What to say (Markdown is stripped; long text is split into sentences).
        language: "yo", "ha", "ig" or "en-ng".
        voice: Optional YarnGPT voice name; empty uses the language's default.
    """
    if tts_engine is None:
        raise gr.Error("Text-to-speech is not enabled on this server.")
    import soundfile as sf  # noqa: PLC0415
    from tts import SAMPLE_RATE  # noqa: PLC0415

    try:
        audio = tts_engine.speak(text, language, voice or None)
    except ValueError as exc:
        raise gr.Error(str(exc)) from exc
    path = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    sf.write(path, audio, SAMPLE_RATE)
    return path


def chat(message: str, history: list[dict]) -> str:
    messages = [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT}]
    messages += [{"role": m["role"], "content": _text(m["content"])} for m in history]
    messages.append({"role": "user", "content": message})
    return generate(messages)


with gr.Blocks(title="N-ATLaS") as demo:
    gr.Markdown("# N-ATLaS\nNCAIR's multilingual model (Yoruba, Hausa, Igbo, English).")
    with gr.Tab("Chat"):
        gr.ChatInterface(chat, api_visibility="private")
    with gr.Tab("API"):
        messages_in = gr.JSON(
            label="messages",
            value=[
                {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
                {"role": "user", "content": "menene ake nufi da gwagwarmaya"},
            ],
        )
        max_tokens_in = gr.Slider(1, MAX_NEW_TOKENS_LIMIT, value=DEFAULT_MAX_NEW_TOKENS, step=1,
                                  label="max_new_tokens")
        reply_out = gr.Textbox(label="reply")
        gr.Button("Generate").click(
            generate, inputs=[messages_in, max_tokens_in], outputs=reply_out, api_name="generate"
        )
    with gr.Tab("Speech"):
        speech_text = gr.Textbox(label="text", value="Ẹ káàbọ̀ sí AgriVoice.")
        speech_lang = gr.Dropdown(["yo", "ha", "ig", "en-ng"], value="yo", label="language")
        speech_voice = gr.Textbox(label="voice (optional)")
        speech_out = gr.Audio(label="speech", type="filepath")
        gr.Button("Speak").click(
            speak, inputs=[speech_text, speech_lang, speech_voice], outputs=speech_out, api_name="speak"
        )

if __name__ == "__main__":
    demo.launch()

In [ ]:
import time
import requests

# How long to keep N-ATLaS online (runs started by hand). Kaggle gives 30 GPU
# hours a week and stops a session after ~9-12 hours anyway.
RUN_HOURS = float(RUN_CONFIG.get("run_hours", 2))
RUN_KEY = RUN_CONFIG.get("run_key")

import app  # downloads N-ATLaS (~16 GB) on first run and loads it onto the GPUs

_, local_url, share_url = app.demo.launch(share=True, prevent_thread_lock=True)
print("\nN-ATLaS public link:", share_url)


def now():
    return time.strftime("%H:%M:%S")


def notify(event):
    """Dashboard-started runs: tell the backend; returns its JSON reply or None."""
    try:
        r = requests.post(
            f"{AGRIVOICE_URL}/api/v1/natlas/{event}",
            json={"run_key": RUN_KEY, "endpoint": share_url},
            timeout=30,
        )
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        print(now(), f"Could not reach AgriVoice ({event}):", exc)
        return None


def register_by_hand():
    if not (AGRIVOICE_URL and AGRIVOICE_ADMIN_TOKEN):
        print("Paste this link into your backend's /admin page.")
        return
    try:
        r = requests.put(
            f"{AGRIVOICE_URL}/api/v1/admin/llm-endpoint",
            json={"endpoint": share_url},
            headers={"Authorization": f"Bearer {AGRIVOICE_ADMIN_TOKEN}"},
            timeout=30,
        )
        r.raise_for_status()
        print(now(), "Registered with AgriVoice:", r.json()["endpoint"])
    except Exception as exc:
        print(now(), "Could not register with AgriVoice:", exc)


stop_at = time.time() + RUN_HOURS * 3600
reason = f"reached {RUN_HOURS:g} h"
try:
    if RUN_KEY:
        notify("register") and print(now(), "Registered with AgriVoice")
        # Check in every minute; the dashboard's Stop button answers "stop"
        while time.time() < stop_at:
            time.sleep(60)
            reply = notify("heartbeat")
            if reply and reply.get("action") == "stop":
                reason = "stopped from the dashboard"
                break
    else:
        register_by_hand()
        # Keep the session alive (and re-register in case the backend restarted)
        while time.time() < stop_at:
            time.sleep(min(600, max(1, stop_at - time.time())))
            register_by_hand()
finally:
    if RUN_KEY:
        notify("stopped")
    app.demo.close()
    print(now(), f"N-ATLaS is now offline ({reason}).")